# Stock Price Predictor — Model Development

This notebook walks through the full ML workflow using **AAPL** as an example:
1. Data acquisition
2. Exploratory data analysis (EDA)
3. Feature engineering & correlation analysis
4. Model training & evaluation
5. Model comparison
6. Key insights

---

## 1. Setup & Data Acquisition

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data_fetcher import fetch_stock_data
from src.feature_engineering import (
    add_technical_indicators, prepare_ml_features,
    get_feature_columns, generate_signals,
)
from src.models import (
    train_random_forest_classifier, train_gradient_boosting_classifier,
    train_linear_regression, train_random_forest_regressor,
    walk_forward_validation,
)
from src.backtester import run_backtest

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup complete.')

### Fetch AAPL data (last 5 years)

In [ ]:
TICKER = 'AAPL'
raw = fetch_stock_data(TICKER, years=5, use_cache=True)
print(f'Shape: {raw.shape}')
raw.tail()

## 2. Exploratory Data Analysis

In [ ]:
# Price history
fig = go.Figure(go.Candlestick(
    x=raw.index, open=raw['Open'], high=raw['High'],
    low=raw['Low'], close=raw['Close'], name='OHLC'))
fig.update_layout(template='plotly_dark', title=f'{TICKER} Price History',
                  height=450, xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# Distribution of daily returns
returns = raw['Close'].pct_change().dropna()
fig = px.histogram(returns, nbins=100, title='Daily Return Distribution',
                   template='plotly_dark')
fig.update_layout(height=350)
fig.show()

print(returns.describe())

In [ ]:
# Summary statistics
raw.describe()

## 3. Feature Engineering

In [ ]:
df_ind = add_technical_indicators(raw)
print(f'Columns after indicators: {len(df_ind.columns)}')
df_ind.tail(3)

In [ ]:
df_ml = prepare_ml_features(df_ind)
feature_cols = get_feature_columns(df_ml)
print(f'Feature columns: {len(feature_cols)}')
print(feature_cols[:15], '...')

### Feature Correlation Matrix

In [ ]:
corr = df_ml[feature_cols].dropna().corr()
fig = px.imshow(corr, color_continuous_scale='RdBu_r',
               title='Feature Correlation Matrix', template='plotly_dark',
               aspect='auto', height=700, width=800)
fig.show()

In [ ]:
# Correlation with target
target_corr = (
    df_ml[feature_cols + ['Target_Direction']]
    .dropna()
    .corr()['Target_Direction']
    .drop('Target_Direction')
    .sort_values()
)
fig = px.bar(x=target_corr.values, y=target_corr.index, orientation='h',
            title='Feature Correlation with Target Direction',
            template='plotly_dark', height=600)
fig.show()

## 4. Model Training & Evaluation

### 4a. Classification — Direction Prediction

In [ ]:
# Random Forest
rf_cls = train_random_forest_classifier(df_ml, feature_cols, test_ratio=0.2)
print('Random Forest Classifier')
print(rf_cls.classification_report_text)
print('Metrics:', rf_cls.metrics)

In [ ]:
# Gradient Boosting
gb_cls = train_gradient_boosting_classifier(df_ml, feature_cols, test_ratio=0.2)
print('Gradient Boosting Classifier')
print(gb_cls.classification_report_text)
print('Metrics:', gb_cls.metrics)

### 4b. Regression — Price Prediction

In [ ]:
# Linear Regression
lr_reg = train_linear_regression(df_ml, feature_cols, test_ratio=0.2)
print('Linear Regression')
print('Metrics:', lr_reg.metrics)

In [ ]:
# Random Forest Regressor
rf_reg = train_random_forest_regressor(df_ml, feature_cols, test_ratio=0.2)
print('Random Forest Regressor')
print('Metrics:', rf_reg.metrics)

## 5. Model Comparison

In [ ]:
# Classification comparison
cls_comparison = pd.DataFrame({
    'Random Forest': rf_cls.metrics,
    'Gradient Boosting': gb_cls.metrics,
}).T
print('=== Classification Models ===')
print(cls_comparison.to_string())

# Regression comparison
reg_comparison = pd.DataFrame({
    'Linear Regression': lr_reg.metrics,
    'Random Forest Regressor': rf_reg.metrics,
}).T
print()
print('=== Regression Models ===')
print(reg_comparison.to_string())

In [ ]:
# Visual comparison
fig = go.Figure()
for name, metrics in [('Random Forest', rf_cls.metrics), ('Gradient Boosting', gb_cls.metrics)]:
    fig.add_trace(go.Bar(name=name, x=list(metrics.keys()), y=list(metrics.values())))
fig.update_layout(template='plotly_dark', title='Classification Model Comparison',
                  barmode='group', height=350, yaxis_range=[0, 1])
fig.show()

### Walk-Forward Validation

In [ ]:
wf = walk_forward_validation(df_ml, feature_cols, n_splits=5)
print(f'Mean walk-forward accuracy: {wf.mean_accuracy:.2%}')
print(f'Per-fold accuracies: {[round(a, 4) for a in wf.accuracies]}')

## 6. Backtest

In [ ]:
bt = run_backtest(
    prices=raw['Close'].reindex(rf_cls.test_dates).dropna(),
    predictions=rf_cls.predictions,
    dates=rf_cls.test_dates,
)
print('Backtest Summary')
for k, v in bt.summary.items():
    print(f'  {k}: {v}')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=bt.portfolio.index, y=bt.portfolio['Strategy'], name='ML Strategy'))
fig.add_trace(go.Scatter(x=bt.portfolio.index, y=bt.portfolio['Buy & Hold'], name='Buy & Hold'))
fig.update_layout(template='plotly_dark', title='Portfolio Value: ML Strategy vs Buy & Hold',
                  yaxis_title='Value ($)', height=400)
fig.show()

## 7. Key Insights

1. **Stock returns are inherently noisy.** Achieving classification accuracy significantly above 50% on next-day direction is challenging and should be interpreted with caution.
2. **Feature importance** consistently highlights momentum and volatility features (e.g., RSI lag, MACD histogram, recent returns) as top predictors.
3. **Walk-forward validation** provides a more realistic estimate of out-of-sample performance than a single train/test split.
4. **Regression models** tend to produce high R-squared values because adjacent-day prices are highly auto-correlated — this does **not** mean the model truly "predicts" price moves.
5. **Backtesting** shows that even modest directional accuracy can outperform buy-and-hold during volatile or bearish periods, but transaction costs and slippage erode returns.

> **Disclaimer:** This analysis is for educational purposes only and does not constitute financial advice.

---
*Built by Eduardo Moraga — [eduardomoraga.github.io](https://eduardomoraga.github.io)*